# ch09 — 산업 적용: BYOD 워크플로

자기 CSV를 즉시 평가하는 전체 흐름.

In [ ]:
import numpy as np


In [ ]:
# 예제 CSV 생성 (실무에서는 자기 데이터 경로 사용)
import pandas as pd

rng = np.random.default_rng(0)
n = 3000
values = np.sin(2*np.pi*np.arange(n)/100) + rng.normal(scale=0.1, size=n)
labels = np.zeros(n, dtype=int); values[2400:2450] += 2.5; labels[2400:2450] = 1
pd.DataFrame({"timestamp": np.arange(n), "value": values, "label": labels}).to_csv(
    "/tmp/my_sensor.csv", index=False)

In [ ]:
from tsad_forge.cli import main

# 라벨 열이 있으므로 전체 지표 산출. 라벨 열이 없으면 스코어+임계값만 출력된다.
main(["run", "--model", "sub_pca", "--data", "/tmp/my_sensor.csv",
      "--results-dir", "/tmp/byod-results"])

In [ ]:
# regime vs fault: 레짐 변화(계절성 전환)는 fault가 아니다 — drift 적응 임계값 비교
from tsad_forge.evaluation.thresholding import dspot_threshold, spot_threshold

drifting = np.arange(3000)/300 + rng.normal(scale=0.5, size=3000)
print("SPOT :", round(spot_threshold(drifting, q=1e-3), 2), "(드리프트를 이상으로 오인)")
print("DSPOT:", round(dspot_threshold(drifting, q=1e-3, depth=100), 2), "(마지막 drift 수준 반영)")